In [ ]:
# read in the dataset

import pandas as pd
import numpy as np

df = pd.read_parquet("modelling_dataset.parquet")
df.head(20)

In [ ]:
df.info()

In [ ]:
import matplotlib.pyplot as plt

# show the distribution of the target variable
# after performing data cleaning, we can see that we have heavy zero inflation
plt.figure(figsize=(10, 5))
plt.hist(df['Accident_Count'], bins=30, edgecolor='black')
plt.xlabel('Accident Count')
plt.ylabel('Frequency')
plt.title('Distribution of Accident Count')
plt.show()

In [ ]:
# from this we see that the variance is ~5.3x the mean (Poisson needs mean=variance) meaning we have overdispersion
print(f"Mean: {round(df['Accident_Count'].mean(), 4)}")
print(f"Variance: {round(df['Accident_Count'].var(), 4)}")
print(f"% zeros: {round((df['Accident_Count'] == 0).mean() * 100, 4)}%")

In [ ]:
import numpy as np

# can't use train_test_split because it randomly shuffles and we need to maintain order due to time-series
d2 = df.sort_values('Time_Stamp')

n = len(df)

# split into 70% train, 15% val, 15% test
train_df = d2.iloc[:int(n*0.7)]
val_df = d2.iloc[int(n*0.7):int(n*0.85)]
test_df = d2.iloc[int(n*0.85):]



In [ ]:
drop_cols = ['Time_Stamp', 'Accident_Count']

# establish X and y for each set, dropping the timestamp and target variable from X
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['Accident_Count']

# do the same for val and test sets
X_val = val_df.drop(columns=drop_cols)
y_val = val_df['Accident_Count']

# and for test set
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['Accident_Count']

In [ ]:
# Poisson predicts a single narrow range for all inputs (if std is far lower than actual std)
# the model has learned no meaningful signal and cannot distinguish high-risk from low-risk zones

from sklearn.linear_model import PoissonRegressor

poisson_model = PoissonRegressor(max_iter=300)
poisson_model.fit(X_train, y_train)

poisson_predictions = poisson_model.predict(X_val)

# lowest and highest risk score the model ever assigns, it should span 0 to 100+ if it were meaningful (it does not: 0.009586 to 0.210420)
print(f"Predicted min: {round(poisson_predictions.min(), 2)}")
print(f"Predicted max: {round(poisson_predictions.max(), 2)}")

# how much the predictions vary across all zones/hours, near zero means the model predicts the same thing everywhere
print(f"Predicted std: {round(poisson_predictions.std(), 2)}")

# how much the actual counts vary, the gap between this and predicted std reveals how blind the model is
print(f"Actual std: {round(y_val.std(), 2)}")

In [ ]:
# the predicted std is 26x smaller than actual std, which means the model assigns nearly identical scores to every row
# regardless of features, meaning it cannot identify high-risk zones at all.
# Since the goal is to identify WHICH zones are dangerous (not predict exact counts),
# we need to reframe this as a classification problem: did an accident occur?